# Notebook de Nettoyage et Préparation des Données
## Projet : Prédiction du Churn E-commerce

Ce notebook couvre toutes les étapes de préparation des données :
1. Chargement et exploration initiale
2. Nettoyage (valeurs manquantes, outliers)
3. Feature Engineering
4. Encodage
5. Split Train/Test + Rééquilibrage SMOTE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration de l'affichage
%matplotlib inline
pd.set_option('display.max_columns', None)

---
## 1. CHARGEMENT ET EXPLORATION INITIALE

In [ ]:
# Chargement du dataset
df = pd.read_excel(r'C:\Users\LENOVO\E-commerceML\E-commerceML\data\raw\retail_customers_COMPLETE_CATEGORICAL.xlsx')

# Identification des colonnes
# Les colonnes numériques : indices 0 à 33 (CustomerID, Recency, ..., SatisfactionScore)
# Les colonnes catégorielles : indices 34 et au-delà (RFMSegment, ...)
cols_numeriques = df.select_dtypes(include=[np.number]).columns.tolist()
cols_categorielles = df.select_dtypes(include=['object']).columns.tolist()

print(f"Nombre de features numériques : {len(cols_numeriques)}")
print(f"Nombre de features catégorielles : {len(cols_categorielles)}")
print(f"Dimensions du dataset : {df.shape}")

# Aperçu des premières lignes
df.head()

In [ ]:
print("--- Structure globale ---")
df.info()

# <class 'pandas.DataFrame'>
# → Indique le type de l'objet : il s'agit d'un DataFrame pandas,
#   autrement dit un tableau de données structuré (lignes × colonnes).

# RangeIndex: 4372 entries, 0 to 4371
# → Représente la taille de la base de données.
#   Ici, nous avons 4 372 observations (clients) indexées de 0 à 4371.

# Data columns (total 52 columns)
# → Le jeu de données contient 52 variables (caractéristiques)
#   décrivant chaque client.

# Non-Null Count
# → Information essentielle pour détecter les valeurs manquantes.
#   Lorsque l'on voit "4372 non-null", cela signifie que la colonne
#   est complète : aucune donnée n'est absente pour ces 4 372 lignes.

# Dtype (Data Type)
# → Indique le type de chaque variable :
#   - int64   : nombres entiers (exemple : 42)
#   - float64 : nombres décimaux (exemple : 12.50)
#   - object  : texte ou variables catégorielles

In [ ]:
# Statistiques descriptives pour détecter les anomalies
# (ex: Recency 0-400, Monetary -5000-15000)
df.describe()

In [ ]:
# Vérification des valeurs manquantes pour les colonnes numériques
print("\n--- Valeurs manquantes (Numériques - Top 10) ---")
print(df[cols_numeriques].isnull().sum().sort_values(ascending=False).head(10))

# Vérification des valeurs manquantes pour les colonnes catégorielles
print("\n--- Valeurs manquantes (Catégorielles) ---")
print(df[cols_categorielles].isnull().sum().sort_values(ascending=False))

# Age : ~1311 valeurs manquantes
# → C'est un volume très important. Sur 4 372 clients, environ 30 % n'ont pas d'âge renseigné.
# → On utilisera l'imputation par la médiane (robuste aux valeurs extrêmes).

# AvgDaysBetweenPurchases : ~79 valeurs manquantes
# → Correspond aux clients ayant effectué une seule commande : intervalle incalculable.
# → On remplacera par 0 (comportement "one-shot").

In [ ]:
# Visualisation de la variable cible : le Churn
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Comptage
sns.countplot(x='Churn', data=df, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribution du Churn (Fidèle vs Parti)')
axes[0].set_xticklabels(['Fidèle (0)', 'Parti (1)'])

# Proportions
churn_pct = df['Churn'].value_counts(normalize=True) * 100
axes[1].pie(churn_pct, labels=['Fidèle (0)', 'Parti (1)'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[1].set_title('Proportion du Churn')

plt.tight_layout()
plt.show()

print(f"\nRépartition : {churn_pct.to_dict()}")
print("→ Déséquilibre détecté → SMOTE sera appliqué sur le Train set.")

---
## 2. NETTOYAGE ET PRÉPARATION DES DONNÉES

In [ ]:
# ================================
# 2.1 Suppression des features inutiles ou à risque de data leakage
# ================================

# IMPORTANT : Les colonnes suivantes doivent être supprimées AVANT la modélisation.
# Raisons :
#   - 'CustomerID' : identifiant unique, sans valeur prédictive.
#   - 'NewsletterSubscribed' : constante (toujours "Yes") → aucune information discriminante.
#   - 'ChurnRiskCategory' : dérivée directement de 'Churn' → DATA LEAKAGE (fausse le modèle).
#   - 'AccountStatus' : liée à la cible (clients partis = compte inactif) → DATA LEAKAGE.
#   - 'LastLoginIP' : sera traitée séparément (extraction d'une feature binaire).

cols_to_drop = ['CustomerID', 'NewsletterSubscribed', 'ChurnRiskCategory', 'AccountStatus']

df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Colonnes supprimées : {cols_to_drop}")
print(f"Dimensions après suppression : {df.shape}")

In [ ]:
# ================================
# 2.2 Imputation de la variable Age (30 % de valeurs manquantes)
# ================================

# On remplace les valeurs manquantes par la médiane.
# La médiane est robuste aux valeurs extrêmes (outliers), contrairement à la moyenne.

from sklearn.impute import SimpleImputer

# Création de l'imputer basé sur la médiane
imputer_age = SimpleImputer(strategy='median')

# fit() : calcule la médiane des âges existants
# transform() : remplace les NaN par cette médiane
df['Age'] = imputer_age.fit_transform(df[['Age']])

# Imputation de AvgDaysBetweenPurchases : remplacer par 0 (clients one-shot)
df['AvgDaysBetweenPurchases'] = df['AvgDaysBetweenPurchases'].fillna(0)

print(f"Valeurs manquantes restantes dans Age : {df['Age'].isnull().sum()}")
print(f"Valeurs manquantes restantes dans AvgDaysBetweenPurchases : {df['AvgDaysBetweenPurchases'].isnull().sum()}")

In [ ]:
# ================================
# 2.3 Traitement des valeurs aberrantes (outliers)
# ================================

# Objectif : corriger les anomalies dans deux indicateurs clés.
# environ 5 à 8 % des données contiennent des valeurs aberrantes (ex : -1, 999, 99).
# Ces anomalies doivent être corrigées pour ne pas fausser les analyses ou la modélisation.

# --- SatisfactionScore ---
# Échelle logique : 1 à 5. Les valeurs hors intervalle (99, -1) sont des erreurs.
if 'SatisfactionScore' in df.columns:
    median_sat = df.loc[(df['SatisfactionScore'] >= 1) & (df['SatisfactionScore'] <= 5), 'SatisfactionScore'].median()
    df.loc[(df['SatisfactionScore'] < 1) | (df['SatisfactionScore'] > 5), 'SatisfactionScore'] = median_sat
    print(f"SatisfactionScore corrigé. Médiane utilisée : {median_sat}")
    print(f"  → Plage après correction : [{df['SatisfactionScore'].min()}, {df['SatisfactionScore'].max()}]")

# --- SupportTicketsCount ---
# Logique métier : 0 à 15 tickets est raisonnable. Au-delà (999 = valeur par défaut), on corrige.
if 'SupportTicketsCount' in df.columns:
    upper_limit = df['SupportTicketsCount'].quantile(0.99)
    df.loc[df['SupportTicketsCount'] > upper_limit, 'SupportTicketsCount'] = upper_limit
    df.loc[df['SupportTicketsCount'] < 0, 'SupportTicketsCount'] = 0
    print(f"SupportTicketsCount corrigé. Limite 99e percentile : {upper_limit}")

---
## 3. FEATURE ENGINEERING

In [ ]:
# ============================================
# 3.1 Parsing et Standardisation de la Date d'inscription
# ============================================

# La colonne 'RegistrationDate' est initialement sous forme de texte.
# On la convertit en format datetime pour pouvoir faire des calculs temporels.

# dayfirst=True  : le format de date est de type jour/mois/année (format UK/Europe).
# errors='coerce': si une date est invalide ou mal écrite, elle est remplacée par NaT.

if 'RegistrationDate' in df.columns:
    reg_date_parsed = pd.to_datetime(
        df['RegistrationDate'],
        dayfirst=True,
        errors='coerce'
    )

    # Extraction des informations temporelles utiles
    # Année d'inscription → permet d'analyser l'ancienneté des clients
    df['RegYear'] = reg_date_parsed.dt.year.fillna(-1).astype(int)

    # Mois d'inscription → utile pour détecter une saisonnalité
    df['RegMonth'] = reg_date_parsed.dt.month.fillna(-1).astype(int)

    # Jour du mois → peut capter certains comportements liés aux cycles de paiement
    df['RegDay'] = reg_date_parsed.dt.day.fillna(-1).astype(int)

    # Jour de la semaine (0 = Lundi, 6 = Dimanche)
    df['RegWeekday'] = reg_date_parsed.dt.weekday.fillna(-1).astype(int)

    # Suppression de la colonne originale
    df.drop(columns=['RegistrationDate'], inplace=True)

    print("RegistrationDate → RegYear, RegMonth, RegDay, RegWeekday créées.")
    print(f"NaN restants dans RegYear : {(df['RegYear'] == -1).sum()}")

In [ ]:
# ============================================
# 3.2 Traitement de l'adresse IP (LastLoginIP)
# ============================================

# On extrait une information utile : l'IP est-elle privée (réseau interne) ou publique ?
# IP privées : 192.168.x.x, 10.x.x.x, 172.16-31.x.x
# Cela peut indiquer si le client utilise un VPN, un réseau d'entreprise, etc.

def is_private_ip(ip):
    """Retourne 1 si l'IP est privée, 0 sinon."""
    try:
        ip_str = str(ip)
        return 1 if ip_str.startswith(('192.168.', '10.', '172.')) else 0
    except:
        return 0

if 'LastLoginIP' in df.columns:
    df['Is_Private_IP'] = df['LastLoginIP'].apply(is_private_ip)
    df.drop(columns=['LastLoginIP'], inplace=True)
    print(f"LastLoginIP → Is_Private_IP créée.")
    print(f"IPs privées : {df['Is_Private_IP'].sum()} / {len(df)}")

In [ ]:
# ============================================
# 3.3 Création de nouvelles features comportementales
# ============================================

# MonetaryPerDay : Panier moyen par jour depuis le dernier achat
if 'MonetaryTotal' in df.columns and 'Recency' in df.columns:
    df['MonetaryPerDay'] = df['MonetaryTotal'] / (df['Recency'] + 1)

# AvgBasketValue : Valeur moyenne d'un panier
if 'MonetaryTotal' in df.columns and 'Frequency' in df.columns:
    df['AvgBasketValue'] = np.where(
        df['Frequency'] > 0,
        df['MonetaryTotal'] / df['Frequency'],
        0
    )

# TenureRatio : Ratio ancienneté / récence (clients récents vs anciens)
if 'CustomerTenureDays' in df.columns and 'Recency' in df.columns:
    df['TenureRatio'] = np.where(
        df['CustomerTenureDays'] > 0,
        df['Recency'] / df['CustomerTenureDays'],
        0
    )

print(f"Nouvelles features créées : MonetaryPerDay, AvgBasketValue, TenureRatio")
print(f"Dimensions actuelles : {df.shape}")

---
## 4. ENCODAGE DES VARIABLES CATÉGORIELLES

In [ ]:
# ============================================
# 4.1 Encodage des variables catégorielles
# ============================================

# On identifie les colonnes catégorielles restantes (après suppression de CustomerID, etc.)
cat_cols_remaining = df.select_dtypes(include=['object']).columns.tolist()
# Retirer 'Churn' si présent (c'est la cible, et elle est souvent déjà numérique)
if 'Churn' in cat_cols_remaining:
    cat_cols_remaining.remove('Churn')

print(f"Colonnes catégorielles à encoder ({len(cat_cols_remaining)}) : {cat_cols_remaining}")

In [ ]:
# One-Hot Encoding avec pd.get_dummies
# drop_first=True pour éviter la multicolinéarité (dummy variable trap)

if cat_cols_remaining:
    df_encoded = pd.get_dummies(df, columns=cat_cols_remaining, drop_first=True)
else:
    df_encoded = df.copy()

print(f"Dimensions avant encodage : {df.shape}")
print(f"Dimensions après encodage : {df_encoded.shape}")
print(f"\nAperçu des nouvelles colonnes :")
print(df_encoded.dtypes.tail(20))

---
## 5. SPLIT TRAIN/TEST + RÉÉQUILIBRAGE SMOTE

In [ ]:
from sklearn.model_selection import train_test_split

# Séparation des variables explicatives (X) et de la cible (y)
# On retire Churn (cible) et les colonnes non-numériques encore présentes
X = df_encoded.select_dtypes(include=[np.number]).drop(columns=['Churn'], errors='ignore')
y = df_encoded['Churn']

print(f"Dimensions de X : {X.shape}")
print(f"Distribution de y :\n{y.value_counts()}")

# Split Train/Test avec STRATIFICATION
# Cela garantit que la proportion de Churn est la même dans Train et Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nDimensions avant rééquilibrage :")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test : {X_test.shape}, y_test : {y_test.shape}")
print(f"\nProportion Churn dans y_train : {y_train.mean():.3f}")
print(f"Proportion Churn dans y_test  : {y_test.mean():.3f}")

In [ ]:
# Rééquilibrage avec SMOTE (Synthetic Minority Over-sampling Technique)
# SMOTE génère des exemples synthétiques de la classe minoritaire (Churn=1)
# IMPORTANT : On applique SMOTE uniquement sur le TRAIN set, jamais sur le TEST set.

try:
    from imblearn.over_sampling import SMOTE

    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    print("--- Résultats du rééquilibrage ---")
    print("Répartition avant SMOTE :\n", y_train.value_counts(normalize=True).round(3))
    print("\nRépartition après SMOTE :\n", y_train_res.value_counts(normalize=True).round(3))
    print(f"\nDimensions après SMOTE : X_train_res={X_train_res.shape}")

except ImportError:
    print("imbalanced-learn non installé. Utilisation du train set original.")
    print("Installez avec : pip install imbalanced-learn")
    X_train_res, y_train_res = X_train, y_train

In [ ]:
import os

# Sauvegarde des données pour la phase de Modélisation
os.makedirs(r'C:\Users\LENOVO\E-commerceML\E-commerceML\data\train_test', exist_ok=True)

# Sauvegarde du train set équilibré
X_train_res_df = pd.DataFrame(X_train_res, columns=X_train.columns)
y_train_res_series = pd.Series(y_train_res, name='Churn')

X_train_res_df.to_csv(
    r'C:\Users\LENOVO\E-commerceML\E-commerceML\data\train_test\X_train_balanced.csv',
    index=False
)
y_train_res_series.to_csv(
    r'C:\Users\LENOVO\E-commerceML\E-commerceML\data\train_test\y_train_balanced.csv',
    index=False
)

# Sauvegarde du test set (NON rééquilibré - il représente la vraie distribution)
X_test.to_csv(
    r'C:\Users\LENOVO\E-commerceML\E-commerceML\data\train_test\X_test_clean.csv',
    index=False
)
y_test.to_csv(
    r'C:\Users\LENOVO\E-commerceML\E-commerceML\data\train_test\y_test_clean.csv',
    index=False
)

# Sauvegarde du dataset nettoyé complet
os.makedirs(r'C:\Users\LENOVO\E-commerceML\E-commerceML\data\processed', exist_ok=True)
df_encoded.to_csv(
    r'C:\Users\LENOVO\E-commerceML\E-commerceML\data\processed\cleaned_encoded_data.csv',
    index=False
)

print("✓ Données sauvegardées :")
print("  - data/train_test/X_train_balanced.csv")
print("  - data/train_test/y_train_balanced.csv")
print("  - data/train_test/X_test_clean.csv")
print("  - data/train_test/y_test_clean.csv")
print("  - data/processed/cleaned_encoded_data.csv")

In [ ]:
# ============================================
# RÉSUMÉ FINAL
# ============================================

print("=" * 55)
print("RÉSUMÉ DU NETTOYAGE")
print("=" * 55)
print(f"Dataset original          : 4372 lignes × 52 colonnes")
print(f"Après nettoyage (encodé)  : {df_encoded.shape[0]} lignes × {df_encoded.shape[1]} colonnes")
print()
print(f"Train set (après SMOTE)   : {X_train_res_df.shape[0]} lignes × {X_train_res_df.shape[1]} colonnes")
print(f"Test set (original)       : {X_test.shape[0]} lignes × {X_test.shape[1]} colonnes")
print()
print("Actions réalisées :")
print("  ✓ Suppression colonnes leakage (ChurnRiskCategory, AccountStatus)")
print("  ✓ Suppression colonnes inutiles (CustomerID, NewsletterSubscribed)")
print("  ✓ Imputation Age par médiane")
print("  ✓ Imputation AvgDaysBetweenPurchases par 0")
print("  ✓ Correction outliers SatisfactionScore et SupportTicketsCount")
print("  ✓ Parsing RegistrationDate → RegYear, RegMonth, RegDay, RegWeekday")
print("  ✓ Extraction IP → Is_Private_IP")
print("  ✓ Feature Engineering : MonetaryPerDay, AvgBasketValue, TenureRatio")
print("  ✓ One-Hot Encoding des variables catégorielles")
print("  ✓ Split Train/Test stratifié (80/20)")
print("  ✓ SMOTE sur le Train set uniquement")